In [0]:
# Spatial libraries needed for the KML -> GeoJSON conversion below
%pip install geopandas shapely

In [0]:
import os
import requests
import geopandas as gpd
import shapely

# 1. Define our Bronze storage paths
BRONZE_DIR = "/Volumes/end-to-end-geospatial-data-pipeline-for-urban-encroachment-analytics/hydraa/bronze"

# 2. GHMC "Hyderabad - Tanks (Lakes)" dataset published on OpenCity (data.opencity.in).
#    This dataset maps the exact polygon perimeters of the region's lakes.
#    It is published in KML format (the only format offered), so we land the raw KML
#    and also write a GeoJSON copy, which is the format the Silver layer expects.
DATA_URL = "https://data.opencity.in/dataset/4becf994-e7f5-4da3-b201-b56b126b57f7/resource/24d7e2e8-74bf-4ba3-bf7c-0867848f74d9/download/9e2e8554-6052-4df3-a8c0-94c993658cc5.kml"
RAW_FILE_PATH = os.path.join(BRONZE_DIR, "hyderabad_lakes_raw.kml")
GEOJSON_FILE_PATH = os.path.join(BRONZE_DIR, "hyderabad_lakes_raw.geojson")

print("🚀 Starting ingestion from OpenCity sources...")

try:
    # Stream the download to prevent memory issues on the free tier instance
    response = requests.get(DATA_URL, stream=True, timeout=30)
    response.raise_for_status()
    
    with open(RAW_FILE_PATH, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)

    # Guard against landing an HTML error page instead of spatial data
    with open(RAW_FILE_PATH, 'rb') as f:
        head = f.read(512)
    if b'<?xml' not in head or b'<kml' not in head:
        raise ValueError("Downloaded content is not KML spatial data (got HTML or another format)")

    print(f"✅ Success! Raw spatial data landed at: {RAW_FILE_PATH}")
    print(f"📦 File Size: {os.path.getsize(RAW_FILE_PATH) / (1024*1024):.2f} MB")

    # Convert the KML polygons layer to the GeoJSON format the pipeline expects
    gdf = gpd.read_file(RAW_FILE_PATH, layer="polygons")
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    gdf.geometry = shapely.force_2d(gdf.geometry.values)  # drop the Z dimension for Sedona compatibility
    gdf.to_file(GEOJSON_FILE_PATH, driver="GeoJSON")
    print(f"🌍 GeoJSON copy written with {len(gdf)} lake polygons at: {GEOJSON_FILE_PATH}")

except Exception as e:
    print(f"❌ Ingestion failed: {str(e)}")


In [0]:
# List the directory contents to confirm the file exists
display(dbutils.fs.ls("dbfs:/Volumes/end-to-end-geospatial-data-pipeline-for-urban-encroachment-analytics/hydraa/bronze"))